# 17. CatBoost under the target-encoded representation

**One variable against ledger row 17** (`lgbm_bag08_seed42_te`, CV 0.966782):
the same nested target and frequency encoder, the same 36 features, the same
five folds, the same seed, the same `learning_rate * iterations = 100` budget
convention. The only thing that changes is the learner.

## Why this is being retried

CatBoost was rejected on 2026-08-04 in `06_catboost.ipynb`. That rejection rested
on two things and the correction of 2026-08-12 removed both of them.

- It was measured **on the raw 12 features**, before target encoding existed. The
  representation that turned out to matter here was not in the test.
- The verdict was a **50/50 rank-average blend**, which scored -0.000250 and
  printed "stop". A combiner with no weights can only average a member in. It
  cannot give it a small weight and it cannot give it a negative one. The neural
  model was rejected the same way and then took a **+0.1178** coefficient in the
  fitted stack, worth a tenth of that stack's total gain.

So the question here is not "is CatBoost as good as LightGBM". It is almost
certainly not, and `06` put that gap at 0.001675 on raw features. The question is
whether a **fitted** combiner can use it, which is the question `06` never asked.

## The gate is pre-registered, and it is not an equal-weight blend

Stated before the run so it cannot be chosen after seeing the answer.

The probe trains one fold and measures CatBoost's **leave-one-out contribution to a
fitted logit stack**, by the split-half protocol from ledger row 24: fit the
combiner on half the held-out rows, score on the other half, five random splits,
paired. The contribution is `AUC(19 members) - AUC(the same 19 without CatBoost)`.

The reference scale is measured **in the same run, on the same rows, by the same
protocol**: the leave-one-out contribution of the **neural model**, the member this
repo previously called the worst blend partner it had. On the full out-of-fold
matrix that is worth +0.000093 (paired sd 0.000003, 5/5 folds), and it is the
smallest contribution in the stack that is still clearly real.

| verdict | rule |
|---|---|
| proceed | positive on >= 4 of 5 splits, mean >= 5e-05, and mean > 2 x paired sd |
| marginal | positive on >= 3 of 5 splits and mean > 0 |
| stop | anything else |

The 5e-05 floor is about half the neural model's recorded contribution. A member
worth less than that does not justify the full five-fold run, which is hours.

## Stages

`SMOKE` exercises every line on 20,000 rows. `RUN_FULL` gates the long run.

1. Load, rebuild the folds, check the fold checksum, re-run the leak checklist.
2. Fingerprint the encoder against `13_target_encoding.ipynb`, then re-run its
   three leak checks by execution.
3. Bench and determinism at 200 iterations, then the fold-0 probe and the gate.
4. Only on `RUN_FULL`: five folds, the out-of-fold vector, the ledger line.


In [ ]:
# Two flags, two runs. Stages 1 to 3 are the probe, which gated stage 4 and is
# re-run here so that one clean kernel produces every number in the ledger row.
SMOKE = False
RUN_FULL = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Fixed rather than -1. NOTES.md: deterministic=True pins a result for a given
# thread count, not across thread counts, and -1 oversubscribed 8 logical cores
# and ran twice as slow. 06 measured CatBoost bit-identical at 6.
THREADS = 6

# The budget convention from experiments 6 to 8, reused by the 06 probe.
LR = 0.05
FULL_ITERS = 2000
BENCH_ITERS = 200
PROBE_FOLD = 0

# Ledger row 17: this feature set, these folds, this seed, LightGBM.
BASELINE_NAME = "lgbm_bag08_seed42_te"
BASELINE_CV = 0.966782
EXPECTED_FOLD_SHA = "ec282b0968059676"

# 13_target_encoding.ipynb printed these. The encoder here must reproduce them.
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

# The row 25 combiner, unchanged. C and max_iter are not tuned here; changing them
# would make this two variables.
STACK_C = 1.0
GATE_SPLITS = 5

# Pre-registered gate. See the header.
GATE_FLOOR = 5e-05

print(f"SMOKE = {SMOKE}   RUN_FULL = {RUN_FULL}")


## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything
trains rather than after.


In [ ]:
import ast
import hashlib
import time
from pathlib import Path

import catboost as cb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold


# Runs here or on Kaggle. Kaggle mounts competition data at
# /kaggle/input/competitions/<slug>/ and datasets at /kaggle/input/datasets/<owner>/,
# one level deeper than the old layout, so both are found by name rather than by
# assuming a shape. Locally the search is restricted to four directories, because
# rglob from the repo root would walk .venv.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist from 06, re-run rather than ticked by inspection. `id` is a
# contiguous row index that separates train from test perfectly, so it is a
# guaranteed leak if it ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back to positions in the saved member vectors, so
# the stacker gate can be exercised in smoke mode too.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    FULL_ITERS, BENCH_ITERS = 100, 50
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set.
A copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse` (which strips comments and formatting but keeps semantics),
and compares checksums. That is the config hash this layout otherwise does not have.


In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")


### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones in `NOTES.md`. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.


In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

# These five encoded frames are about 300 MB and nothing below reads them. A first
# attempt at this run was killed partway through on a machine with 3.7 GB free, and
# while that was never diagnosed, holding them to the end of a two hour run is a
# cost with no benefit either way.
import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()


## Stage 3. CatBoost, bench and determinism

CatBoost gets the identical 36 columns LightGBM got. The three categoricals stay in
as native categoricals with missing as its own level, exactly as `06` had them, and
their target and frequency encodings are added alongside, exactly as row 17 had
them. That is duplicative, and it is duplicative for LightGBM too, which is what
makes this one variable rather than two.

Numeric NaN is left to CatBoost's native handling, which is the counterpart of
LightGBM's native routing. No imputation, per the null result in ledger row 18.


In [ ]:
CAT_IDX = None


def to_cb(df):
    """CatBoost wants categoricals as strings with no NaN."""
    d = df.copy()
    for c in CAT:
        d[c] = d[c].astype("object").fillna("__NA__").astype(str)
    return d


def run_fold(fold, iters, seed=SEED, verbose=0, want_test=False):
    """Encode inside the fold, train on the rest, predict the fold."""
    global CAT_IDX
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test if want_test else None)
    Xtr, Xva = to_cb(Xtr), to_cb(Xva)
    CAT_IDX = [Xtr.columns.get_loc(c) for c in CAT]

    m = cb.CatBoostClassifier(
        iterations=iters, learning_rate=LR, random_seed=seed,
        thread_count=THREADS, allow_writing_files=False, verbose=verbose,
    )
    t0 = time.time()
    m.fit(Xtr, y[tr], cat_features=CAT_IDX)
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(to_cb(Xte))[:, 1] if want_test else None
    return {"p": p, "va": va, "secs": secs, "p_te": p_te,
            "auc": float(roc_auc_score(y[va], p))}


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "17_catboost_te.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def mem():
    """Free physical memory in GB, or None where it cannot be read."""
    try:
        if not hasattr(__import__("os"), "add_dll_directory"):
            for line in Path("/proc/meminfo").read_text().splitlines():
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 2 ** 20
            return None
        import ctypes

        class MS(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong),
                        ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong),
                        ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong),
                        ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong),
                        ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]

        m = MS()
        m.dwLength = ctypes.sizeof(MS)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
        return m.ullAvailPhys / 2 ** 30
    except Exception:
        return None


def note(msg):
    """Print, and append to a log file flushed on every write.

    nbconvert writes the notebook only once the whole run finishes, so without
    this there is no way to watch a run that takes over an hour from outside the
    kernel. Carried over from 06, which learned it the hard way. The free-memory
    stamp is here because the first attempt at the full run was killed partway
    through and there was no evidence either way about why.
    """
    print(msg)
    g = mem()
    tail = "" if g is None else f"  [{g:.1f} GB free]"
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}{tail}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, RUN_FULL={RUN_FULL}, threads={THREADS} ===")

a = run_fold(PROBE_FOLD, BENCH_ITERS)
b = run_fold(PROBE_FOLD, BENCH_ITERS)
delta = float(np.abs(a["p"] - b["p"]).max())
DETERMINISTIC = delta == 0.0

per_100 = a["secs"] / BENCH_ITERS * 100
proj_probe = per_100 * FULL_ITERS / 100

print(f"catboost {cb.__version__}, {THREADS} threads, {len(CAT_IDX)} categoricals "
      f"of {len(a['p']) and 'n/a'}".replace(" of n/a", ""))
print(f"{BENCH_ITERS} iters on fold {PROBE_FOLD}: {hhmm(a['secs'])}, "
      f"AUC {a['auc']:.6f}")
print(f"determinism, two identical runs, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")
print()
print(f"rate {per_100:.1f}s per 100 iterations")
print(f"  stage 3 probe, 1 fold x {FULL_ITERS}: {hhmm(proj_probe)}")
print(f"  stage 4 full,  5 folds x {FULL_ITERS}: {hhmm(proj_probe * 5)}"
      f"   (plus test prediction, which the probe does not do)")
note(f"stage 2 done, rate {per_100:.1f}s/100 iters, determinism "
     f"{'OK' if DETERMINISTIC else 'FAILED'}")


### The fold-0 probe

One fold at the full budget. Two numbers come out of it: how far behind LightGBM
CatBoost is under this representation, which is description, and what it is worth
to a fitted combiner, which is the decision.


In [ ]:
probe = run_fold(PROBE_FOLD, FULL_ITERS, verbose=max(FULL_ITERS // 5, 1))
cat_p = probe["p"]
gate_rows = probe["va"]
yg = y[gate_rows]

te42 = np.load(locate("te_bag42_oof.npy"))[ROW_IDX][gate_rows]
auc_te42 = float(roc_auc_score(yg, te42))

print()
print(f"fold {PROBE_FOLD}, {len(gate_rows):,} rows, both scored on the same rows")
print(f"  CatBoost, {FULL_ITERS} iters : {probe['auc']:.6f}   ({hhmm(probe['secs'])})")
print(f"  LightGBM, row 17            : {auc_te42:.6f}")
print(f"  gap                         : {probe['auc'] - auc_te42:+.6f}")
print()
print("06 measured this gap at -0.001675 on the raw 12 features. It is description,")
print("not the verdict: the neural model is 0.0247 behind and still earns a weight.")
note(f"stage 3 probe done, fold {PROBE_FOLD} AUC {probe['auc']:.6f}")


In [ ]:
# The 18 members of the row 25 stack, in its order. CatBoost joins as the 19th.
MEM = [
    ("te42", "te_bag42_oof.npy"), ("te2024", "te_seed2024_oof.npy"),
    ("te7", "te_seed7_oof.npy"), ("te2025", "te_seed2025_oof.npy"),
    ("te13", "te_seed13_oof.npy"),
    ("anchor", "lgbm_default_anchor_seed42.npy"),
    ("trees300", "lgbm_trees300_seed42.npy"),
    ("trees1000", "lgbm_trees1000_seed42.npy"),
    ("trees2000", "lgbm_trees2000_seed42.npy"),
    ("lr010", "lgbm_lr01_n1000_seed42.npy"),
    ("lr005", "lgbm_lr005_n2000_seed42.npy"),
    ("lr003", "lgbm_lr003_n3333_seed42.npy"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42.npy"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024.npy"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7.npy"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025.npy"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13.npy"),
    ("neural", "neural_oof.npy"),
]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


names = [n for n, _ in MEM] + ["catboost_te"]
cols_all = list(range(len(names)))
Lg = np.column_stack(
    [logit(np.load(locate(f))[ROW_IDX][gate_rows]) for _, f in MEM]
    + [logit(cat_p)])
# A partially failed run leaves a constant column, which blends silently.
assert Lg.shape == (len(gate_rows), 19) and np.ptp(Lg, axis=0).min() > 0

print(f"gate matrix {Lg.shape} on fold {PROBE_FOLD}'s rows")
print("member AUC on these rows:")
for i, n in enumerate(names):
    print(f"  {n:12} {roc_auc_score(yg, Lg[:, i]):.6f}")


In [ ]:
# The gate. Split-half within fold 0, the protocol row 24 used for its honest
# number, five random splits, paired. Every one of these 19 vectors is out-of-fold
# for these rows, so a combiner fit on half of them and scored on the other half
# never sees a row's own target through any route.
def stack_auc(cols, tr, va):
    clf = LogisticRegression(C=STACK_C, max_iter=2000).fit(Lg[np.ix_(tr, cols)],
                                                           yg[tr])
    return roc_auc_score(yg[va], clf.decision_function(Lg[np.ix_(va, cols)]))


drop = {n: [i for i in cols_all if names[i] != n] for n in ("catboost_te", "neural")}
rng = np.random.default_rng(SEED)
g_cat, g_neu, base18 = [], [], []

for s in range(GATE_SPLITS):
    perm = rng.permutation(len(gate_rows))
    tr, va = perm[:len(perm) // 2], perm[len(perm) // 2:]
    full = stack_auc(cols_all, tr, va)
    without_cat = stack_auc(drop["catboost_te"], tr, va)
    g_cat.append(full - without_cat)
    g_neu.append(full - stack_auc(drop["neural"], tr, va))
    base18.append(without_cat)
    print(f"  split {s}: 18-member {without_cat:.6f}  19-member {full:.6f}  "
          f"catboost {g_cat[-1]:+.6f}")

g_cat, g_neu = np.array(g_cat), np.array(g_neu)
print()
print(f"{'member':12} {'mean':>10} {'paired sd':>11} {'splits won':>11}")
for lbl, g in (("catboost_te", g_cat), ("neural", g_neu)):
    print(f"{lbl:12} {g.mean():>+10.6f} {g.std(ddof=1):>11.6f} "
          f"{(g > 0).sum():>8}/{GATE_SPLITS}")
print()
print("`neural` is the in-run reference scale, not a second result: it is the")
print("weakest member that still clearly earns its place, worth +0.000093 on the")
print("full out-of-fold matrix in row 25.")


In [ ]:
mean_g, sd_g, wins = g_cat.mean(), g_cat.std(ddof=1), int((g_cat > 0).sum())

if not (LEAK_OK and CLEAN):
    VERDICT = "blocked"
    why = "a leak check failed, so nothing measured above is safe to act on"
elif not ENCODER_MATCH:
    VERDICT = "blocked"
    why = "the encoder does not match 13, so this is not one variable against row 17"
elif not DETERMINISTIC:
    VERDICT = "blocked"
    why = "the configuration is not reproducible, so the gain cannot be separated \
from run-to-run noise"
elif wins >= 4 and mean_g >= GATE_FLOOR and mean_g > 2 * sd_g:
    VERDICT = "proceed"
    why = ("the fitted combiner pays for CatBoost on held-out rows, above the floor "
           "and above its own noise. Set RUN_FULL = True and run all.")
elif wins >= 3 and mean_g > 0:
    VERDICT = "marginal"
    why = ("positive but under the pre-registered bar. Record it as inconclusive "
           "rather than as an improvement, per CLAUDE.md.")
else:
    VERDICT = "stop"
    why = ("a fitted combiner does not want it either, which is the test 06 never "
           "ran. That closes CatBoost on grounds that survive the 2026-08-12 "
           "correction, and XGBoost with it.")

print(f"gate:  mean {mean_g:+.6f}   floor {GATE_FLOOR:+.6f}   "
      f"2 x paired sd {2 * sd_g:.6f}   wins {wins}/{GATE_SPLITS}")
print()
print(f"VERDICT: {VERDICT}")
print(f"  {why}")


## Stage 4. The full run

Skipped unless `RUN_FULL` is on. Five folds at the full budget, the encoder rebuilt
inside each one, the test set predicted by every fold model and averaged, which is
what every other submission in this repo does.


In [ ]:
full = None
if not RUN_FULL:
    print("stage 4 skipped, RUN_FULL is False. The probe above is this run's output.")
else:
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    scores = []
    t0 = time.time()
    for f in range(5):
        r = run_fold(f, FULL_ITERS, want_test=True)
        oof[r["va"]] = r["p"]
        test_pred += r["p_te"] / 5
        scores.append(r["auc"])
        done = time.time() - t0
        note(f"  fold {f}: {scores[-1]:.6f}  ({hhmm(r['secs'])}), "
             f"elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")

    cv, sd = float(np.mean(scores)), float(np.std(scores))
    full = {"cv": cv, "sd": sd, "oof": oof, "test": test_pred, "scores": scores}
    print()
    print(f"CatBoost CV {cv:.6f} +/- {sd:.6f}   [{hhmm(time.time() - t0)}]")

    # Fold PROBE_FOLD was trained twice in this kernel, once in stage 3 and once
    # here. Determinism was checked at 200 iterations; this checks it at the full
    # budget and across the whole run, for free.
    repeat = abs(scores[PROBE_FOLD] - probe["auc"])
    print(f"fold {PROBE_FOLD} retrained at the full budget: {repeat:.3e} from the "
          f"probe  {'OK' if repeat == 0.0 else 'DIVERGED'}")

    # Paired against ledger row 17 on the identical folds. The right bar for
    # "is B better than A" is the spread of the per-fold differences and how many
    # folds it wins, not the fold spread, which is common to both and cancels.
    base = np.load(locate("te_bag42_oof.npy"))[ROW_IDX]
    bf = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(5)])
    d = np.array(scores) - bf
    print(f"row 17 reproduces its ledger number to {bf.mean() - BASELINE_CV:+.2e}")
    print(f"per-fold differences vs row 17: {np.round(d, 6).tolist()}")
    print(f"mean {d.mean():+.6f}, paired sd {d.std(ddof=1):.6f}, "
          f"wins {(d > 0).sum()}/5 folds")


In [ ]:
if full is not None:
    pre = "SMOKE_" if SMOKE else ""
    np.save(OUT / f"{pre}catboost_te_oof.npy", full["oof"])
    np.save(OUT / f"{pre}catboost_te_test.npy", full["test"])
    pd.DataFrame({"id": test["id"], TARGET: full["test"]}).to_csv(
        SUB / f"{pre}catboost_te.csv", index=False)
    print(f"wrote {pre}catboost_te_oof.npy, {pre}catboost_te_test.npy, "
          f"{pre}catboost_te.csv")
    print()
    print("ledger line:")
    print(f"  name    catboost_te")
    print(f"  cv_mean {full['cv']:.6f}")
    print(f"  cv_std  {full['sd']:.6f}")
    print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
          f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
          f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}")
    print()
    print("The stack refit is 18_stack_19.ipynb, and it is a separate ledger row")
    print("because it changes a different variable.")
